In [ ]:
##Import the necessary libraries
import pandas as pd
import numpy as np
import os
import matplotlib.pyplot as plt
import seaborn as sns
import geopandas as gpd

import warnings
warnings.simplefilter(action='ignore', category=UserWarning)
warnings.simplefilter(action='ignore', category=RuntimeWarning)
pd.options.mode.chained_assignment = None

In [ ]:
# Set the working directory where input files are stored in 'input' folder, and output files will be saved in 'output' folder. It is essential to have the 2 folders in the working directory
os.chdir(r"C:\Users\...")

In [ ]:
def post_process_scenario_data(scenario_sed, scenario_naics, run_id):
    """
    Processes scenario_sed and scenario_naics, aggregates by TAZ, and generates a final dataset.
    
    Args:
        scenario_sed (pd.DataFrame): Socioeconomic data for the scenario.
        scenario_naics (pd.DataFrame): Employment data for the scenario.
        run_id (int): Run ID for naming the final output file.

    """

    # Load the MAZ lookup table from DBF
    #maztaz = gpd.read_file("MAZ_LOOKUP.dbf")

    # Load additional employment data
    input_folder = "inputs"
    output_folder = "output"
    EMPpt_file = os.path.join(input_folder, "EMPFORTRUCK.csv")
    EMPpt = pd.read_csv(EMPpt_file)

    # Rename columns for consistency
    EMPpt['TAZ'] = EMPpt['Taz']

    # Filter retail employment (NAICS codes 44 and 45)
    RETAILEMP = EMPpt[(~EMPpt['TAZ'].isna()) & ((EMPpt['Naics2'] == 44) | (EMPpt['Naics2'] == 45))]

    # Replace missing SQFT values with 0
    RETAILEMP.loc[:, 'FCOMSQFT'] = RETAILEMP['SQFT'].fillna(0)

    # Summarize socioeconomic data by TAZ
    SETAZ = scenario_sed.groupby('TAZ')[['RESPOP', 'RESHH']].sum().reset_index()
    # Rename columns for clarity
    SETAZ.rename(columns={'RESPOP': 'TOTPOP', 'RESHH': 'TOTHH'}, inplace=True)

    # Summarize employment data by TAZ
    JOBTAZ = scenario_naics.groupby('TAZ').agg(
        TOTALEMP=('Total', 'sum'),
        FARME=('n11', 'sum'),
        MININGE=('n21', 'sum'),
        CONSTRUCTI=('n23', 'sum'),
        WHOLESALEE=('n42', 'sum')
    ).reset_index()
    JOBTAZ['RETAILE'] = scenario_naics.groupby('TAZ')[['n44', 'n45']].sum().sum(axis=1).values
    JOBTAZ['MANUFACTUR'] = scenario_naics.groupby('TAZ')[['n31', 'n32', 'n33']].sum().sum(axis=1).values
    service_cols = ['n51', 'n52', 'n53', 'n54', 'n55', 'n56', 'n61', 'n62', 'n71', 'n72', 'n81', 'n92']
    JOBTAZ['SERVICEE'] = scenario_naics.groupby('TAZ')[service_cols].sum().sum(axis=1).values

    # Summarize retail square footage by TAZ
    SQFTTAZ = RETAILEMP.groupby('TAZ').agg(RETAILE_SQ=('FCOMSQFT', 'sum')).reset_index()

    # Add new columns and compute transformations
    SETAZ['TOTPOP1'] = SETAZ['TOTPOP']
    SETAZ['LNTOTHH'] = np.round(np.log(SETAZ['TOTHH'].replace(0, np.nan)), 0).fillna(0)

    # Replace infinite values with 0 (handling log(0) cases)
    SETAZ['LNTOTHH'].replace([np.inf, -np.inf], 0, inplace=True)

    # Merge datasets
    FINAL = SETAZ.merge(JOBTAZ, on="TAZ", how="left")
    FINAL = FINAL.merge(SQFTTAZ, on="TAZ", how="left")

    # Replace missing square footage values with 0
    FINAL['RETAILE_SQ'] = FINAL['RETAILE_SQ'].fillna(0)

    # Load SED sample structure to match column names
    SEDSAMPLE_file = os.path.join(input_folder, "SED_DATA_sample.csv")
    SEDSAMPLE = pd.read_csv(SEDSAMPLE_file)
    FINAL['YEAR'] = 2055

    # Ensure only relevant columns are included in FINAL
    FINAL = FINAL[SEDSAMPLE.columns]

    # Compute SQFT_JOB ratio
    SQFT_JOB = FINAL['RETAILE_SQ'].sum() / FINAL['RETAILE'].sum()

    # Assign missing square footage where applicable
    mask = (FINAL['RETAILE_SQ'] == 0) & (FINAL['RETAILE'] > 0)
    FINAL.loc[mask, 'RETAILE_SQ'] = FINAL.loc[mask, 'RETAILE'] * SQFT_JOB

    # Define output filename
    output_filename = f"SEDATAPAG_2055_{run_id}.csv"

    # Save to CSV
    FINAL.to_csv(os.path.join(output_folder, output_filename), index=False)



In [ ]:
"""PAG experimental design code
This code interpolates socioeconomic and employment data across different growth and distribution scenarios, adjusts household 
and population categories to maintain consistency, and ensures totals align with expected ranges before saving scenario outputs and logging quality checks.

    Args:
        scenario_sed (pd.DataFrame): Socioeconomic data for the scenario.
        scenario_naics (pd.DataFrame): Employment data for the scenario.
        run_id (int): Run ID for naming the final output file.
    
    Saves:
        scenario_sed (csv): Interpolated socioeconomic data for the scenario at MAZ level.
        scenario_naics (csv): Interpolated employment data for the scenario at MAZ level.
        summary_file (csv): Summary file with total population, households, and employment for each scenario.
        qaqc_file (csv): QAQC file with selected MAZ data for verification.
        SEDATA (csv): Final output file with aggregated data by TAZ."""

"""
1. Setup & Initialization

-Defines input/output directories.
-Loads input parameters from an Excel file.
-Loads socioeconomic and employment datasets for four scenarios (LoLo, LoHi, HiLo, HiHi).

2. Preprocessing Data

-Computes population-to-household ratios (PopHH_ratio).
-Calculates population category shares for each scenario.

3. Interpolation Functions

-Defines functions to interpolate socioeconomic and employment data based on growth and distribution weights.

4. Scenario Generation

-For each scenario run, interpolates household income, structure type, household size, and workforce categories.
-Adjusts rounding discrepancies to maintain data consistency.
-Computes revised total population and distributes population across categories.
-Interpolates employment data across multiple sectors.

5. Scenario Output & Validation

-Saves the interpolated socioeconomic and employment data as CSV files.
-Updates the summary file with total population, households, employment, and categorical sums.

6. QAQC Process

-Extracts specific MAZ (zone) data for verification.
-Stores household density, income categories, and employment values for selected MAZs.
-Logs all scenario results into a QAQC file.

7. Finalization

-Saves the QAQC file.
-Prints execution completion messages."""


# Define input and output folder paths
input_folder = "inputs"
output_folder = "output"
summary_file = os.path.join(output_folder, "scenario_summary.csv")

# Ensure the output folder exists
os.makedirs(output_folder, exist_ok=True)

# Load the input parameters from the Excel file
param_file = os.path.join(input_folder, "pag_emat_design_experiments.csv")
params = pd.read_csv(param_file)

# Load socioeconomic (socec_maz) and employment (naics_maz) data files
high_high_sed = pd.read_csv(os.path.join(input_folder, 'year_2055_socec_mazHiGrHiCap.csv'))
low_high_sed = pd.read_csv(os.path.join(input_folder, 'year_2055_socec_mazLoGrHiCap.csv'))
high_low_sed = pd.read_csv(os.path.join(input_folder, 'year_2055_socec_mazHiGrLoCap.csv'))
low_low_sed = pd.read_csv(os.path.join(input_folder, 'year_2055_socec_mazLoGrLoCap.csv'))

high_high_naics = pd.read_csv(os.path.join(input_folder, 'year_2055_naics_mazHiGrHiCap.csv'))
low_high_naics = pd.read_csv(os.path.join(input_folder, 'year_2055_naics_mazLoGrHiCap.csv'))
high_low_naics = pd.read_csv(os.path.join(input_folder, 'year_2055_naics_mazHiGrLoCap.csv'))
low_low_naics = pd.read_csv(os.path.join(input_folder, 'year_2055_naics_mazLoGrLoCap.csv'))

# List of employment sectors contributing to TOTALEMP
employment_sectors = ['n11', 'n21', 'n22', 'n23', 'n31', 'n32', 'n33', 'n42', 'n44', 'n45',
    'n48', 'n49', 'n51', 'n52', 'n53', 'n54', 'n55', 'n56', 'n61', 'n62',
    'n71', 'n72', 'n721', 'n722', 'n81', 'n92', 'n928']

# Population categories
pop_categories = ['POP0_5', 'POP6_12', 'POP13_15', 'POP16_18', 'POP19_25', 
                  'POP26_35', 'POP36_45', 'POP46_55', 'POP56_65', 'POP66_80', 'POP81']

# Household sub-categories (each sums to RESHH)
hhinc_categories = ['HHINC1', 'HHINC2', 'HHINC3', 'HHINC4', 'HHINC5']
sfmf_hh_categories = ['SFHH', 'MFHH']
hhsize_categories = ['HHSIZE1', 'HHSIZE2', 'HHSIZE3', 'HHSIZE4', 'HHSIZE5', 'HHSIZE6', 'HHSIZE7']
hhwork_categories = ['HHWORK0', 'HHWORK1', 'HHWORK2', 'HHWORK3']

popHH_ratio_boundaries = [2.385559431, 2.395770068, 2.408858546, 2.443903556]


def preprocess_socioeconomic_data(df):
    df['PopHH_ratio'] = df['RESPOP'] / df['RESHH']
    return df

def compute_population_shares(df):
    """Computes the share of each population category relative to total RESPOP"""
    shares = df[pop_categories].div(df["RESPOP"], axis=0).fillna(0)
    return shares

# Apply preprocessing to all input datasets
low_low_sed = preprocess_socioeconomic_data(low_low_sed)
low_high_sed = preprocess_socioeconomic_data(low_high_sed)
high_low_sed = preprocess_socioeconomic_data(high_low_sed)
high_high_sed = preprocess_socioeconomic_data(high_high_sed)

pop_shares_lolo = compute_population_shares(low_low_sed)
pop_shares_lohi = compute_population_shares(low_high_sed)
pop_shares_hilo = compute_population_shares(high_low_sed)
pop_shares_hihi = compute_population_shares(high_high_sed)


# Interpolation function

def interpolate_socioeconomic_data(low, high, weight):
    return low * (1 - weight) + high * weight

def interpolate_columns(low_low, high_low, low_high, high_high, growth_weight, distribution_weight):
    return interpolate_socioeconomic_data(
        interpolate_socioeconomic_data(low_low, high_low, growth_weight),
        interpolate_socioeconomic_data(low_high, high_high, growth_weight),
        distribution_weight)

# Function to generate scenario data with density-based redistribution
def generate_scenario_data(growth_weight, distribution_weight, run_id):
    """Interpolates each category individually and adjusts using density factors."""

    # Step 1: Interpolate Household Categories
    """ The code interpolates household category data (income, structure type, size, and work status) based on the given growth and distribution weights.
        It loops through four household categories (hhinc, sfmf, hhsize, hhwork), applying weighted interpolation to estimate values for each subcategory.
        Finally, it computes the total number of households (RESHH) by summing the household income subcategories."""
    final_hh = {}
    for category, columns in zip(
        ['hhinc', 'sfmf', 'hhsize', 'hhwork'],
        [hhinc_categories, sfmf_hh_categories, hhsize_categories, hhwork_categories]
    ):
        interpolated_hh = {col: interpolate_columns(
            low_low_sed[col], high_low_sed[col], low_high_sed[col], high_high_sed[col], growth_weight, distribution_weight
        ) for col in columns}
        final_hh[category] = pd.DataFrame(interpolated_hh).round()

    new_reshh = final_hh['hhinc'].sum(axis=1).round()

    # Step 2: Adjust Other Household Categories to Match new_reshh
    """After interpolation, the total number of households (RESHH) may change.
        Subcategories must be scaled proportionally to maintain internal consistency.
        Ensures the sum of all household subcategories exactly matches new_reshh, preventing discrepancies in downstream calculations."""
    for category in ['sfmf', 'hhsize', 'hhwork']:
        category_total = final_hh[category].sum(axis=1)  # Compute the total per row
        scaling_factor = new_reshh / category_total.replace(0, np.nan)  # Avoid division by zero
        adjusted_hh = (final_hh[category].mul(scaling_factor, axis=0)).round().fillna(0)  # Scale and round
        #Distribute Rounding Errors to Match Exact Total
        discrepancy = new_reshh - adjusted_hh.sum(axis=1)  # Compute the difference
        adjustment_column = adjusted_hh.columns[-1]  # Pick the last column to absorb rounding errors
        adjusted_hh[adjustment_column] += discrepancy  # Apply the adjustment
        final_hh[category] = adjusted_hh

    # Step 3: Interpolate Population/Household ratios
    """Interpolate PopHH_ratio using weighted averages from different scenarios.
        Estimate initial total population by multiplying PopHH_ratio with new_reshh.
        Determine expected total population based on predefined ratio boundaries.
        Scale population to match the expected total, ensuring consistency.""" 
    popHH_ratio = interpolate_socioeconomic_data(interpolate_socioeconomic_data(low_low_sed['PopHH_ratio'], high_low_sed['PopHH_ratio'], growth_weight),
                                                        interpolate_socioeconomic_data(low_high_sed['PopHH_ratio'], high_high_sed['PopHH_ratio'], growth_weight), distribution_weight)
    total_pop_revised = (popHH_ratio.mul(new_reshh, axis=0)).round()

    total_pop_range = [new_reshh.sum() * m for m in popHH_ratio_boundaries]
    expected_total_pop = interpolate_socioeconomic_data(interpolate_socioeconomic_data(total_pop_range[0], total_pop_range[2], growth_weight),
                                                        interpolate_socioeconomic_data(total_pop_range[1], total_pop_range[3], growth_weight), distribution_weight)

    scaling_factor_pop = expected_total_pop / total_pop_revised.sum()
    scaled_population = total_pop_revised * scaling_factor_pop
    final_total_pop = scaled_population.round().fillna(0)

        # Step 3.1: Compute Population Shares and Interpolation
    interpolated_pop_shares = {col: interpolate_columns(
        pop_shares_lolo[col], pop_shares_hilo[col], pop_shares_lohi[col], pop_shares_hihi[col], growth_weight, distribution_weight
    ) for col in pop_categories}
    interpolated_pop_shares = pd.DataFrame(interpolated_pop_shares)
    row_sums = interpolated_pop_shares.sum(axis=1)
    normalized_pop_shares = interpolated_pop_shares.div(row_sums, axis=0).fillna(0)

        # Step 3.2: Apply Population Shares to Total Population
    final_pop_categories = (normalized_pop_shares.mul(final_total_pop, axis=0)).round()
    adjustment = final_total_pop - final_pop_categories.sum(axis=1)  # Compute discrepancy
    adjustment_column = final_pop_categories.columns[-1] 
    final_pop_categories[adjustment_column] += adjustment.fillna(0)  # Distribute rounding errors

    # Step 4: Interpolate Employment Categories
    """Interpolate employment data for each sector using weighted averages from different scenarios.
    Convert interpolated values into a DataFrame.
    Round values to maintain realistic employment counts."""
    interpolated_emp = {col: interpolate_columns(
        low_low_naics[col], high_low_naics[col], low_high_naics[col], high_high_naics[col], growth_weight, distribution_weight
    ) for col in employment_sectors}
    interpolated_emp = pd.DataFrame(interpolated_emp)
    final_emp = interpolated_emp.round()


    # Step 5: Compute final Totals
    new_total_pop = final_total_pop.round().fillna(0)
    new_total_hh = new_reshh.round()
    new_total_emp = final_emp.sum(axis=1).round()

    # Step 7: Create Output DataFrames
    scenario_sed = low_low_sed.copy()
    scenario_sed[pop_categories] = final_pop_categories.fillna(0)
    scenario_sed['RESHH'] = new_total_hh
    scenario_sed['RESPOP'] = new_total_pop.fillna(0)
    for cat in final_hh:
        scenario_sed[final_hh[cat].columns] = final_hh[cat]

    scenario_naics = low_low_naics.copy()
    scenario_naics[employment_sectors] = final_emp
    scenario_naics['Total']= new_total_emp

        # Step 7.1: Compute sums for different categories
    pop_sum = final_total_pop.sum()
    hhinc_sum = final_hh['hhinc'].sum().sum()
    sfmf_sum = final_hh['sfmf'].sum().sum()
    hhsize_sum = final_hh['hhsize'].sum().sum()
    hhwork_sum = final_hh['hhwork'].sum().sum()

    return scenario_sed, scenario_naics, new_total_pop.sum(), new_total_hh.sum(), new_total_emp.sum(), pop_sum, hhinc_sum, sfmf_sum, hhsize_sum, hhwork_sum

# Step 7: Record Original Scenario Totals (LoLo, LoHi, HiLo, HiHi) for summary
original_totals = pd.DataFrame([
    {"Scenario": "LoLo", "Total Population": low_low_sed["RESPOP"].sum(), 
     "Total Households": low_low_sed["RESHH"].sum(), "Total Employment": low_low_naics["Total"].sum()},
    {"Scenario": "LoHi", "Total Population": low_high_sed["RESPOP"].sum(), 
     "Total Households": low_high_sed["RESHH"].sum(), "Total Employment": low_high_naics["Total"].sum()},
    {"Scenario": "HiLo", "Total Population": high_low_sed["RESPOP"].sum(), 
     "Total Households": high_low_sed["RESHH"].sum(), "Total Employment": high_low_naics["Total"].sum()},
    {"Scenario": "HiHi", "Total Population": high_high_sed["RESPOP"].sum(), 
     "Total Households": high_high_sed["RESHH"].sum(), "Total Employment": high_high_naics["Total"].sum()}
])
original_totals.to_csv(summary_file, index=False)

    # Step 7.1: Randomly select desired number of unique MAZs for verification
selected_mazs= [51930,	55885,	52757,	54741,	51360]
qaqc_file = os.path.join(output_folder, "qaqc_hh_density.csv")
# Create a DataFrame to store QAQC results
qaqc_results = pd.DataFrame(columns=['Run'] + list(selected_mazs) + ['Density_weight'] + list(selected_mazs) + list(selected_mazs) + list(selected_mazs) 
                            + list(selected_mazs) + list(selected_mazs)
                            + list(selected_mazs)+ list(selected_mazs))

# Step 8: Run the main loop through scenarios & log results**
for index, row in params.iterrows():
    run_id = int(row['experiment'])
    growth_weight, distribution_weight = row['HH_EMP_Growth'], row['HH_EMP_Distribution']

    scenario_sed, scenario_naics, total_pop, total_hh, total_emp, pop_sum, hhinc_sum, sfmf_sum, hhsize_sum, hhwork_sum = generate_scenario_data(growth_weight, distribution_weight, run_id)
    scenario_sed.drop(columns=['PopHH_ratio'], inplace=True)
    # Save scenario output
    sed_filename = f"year_2055_socec_maz_scenario_run{run_id}.csv"
    naics_filename = f"year_2055_naics_maz_scenario_run{run_id}.csv"

    scenario_naics.to_csv(os.path.join(output_folder, naics_filename), index=False)
    scenario_sed.to_csv(os.path.join(output_folder, sed_filename), index=False)
    post_process_scenario_data(scenario_sed, scenario_naics, run_id)

    new_summary_row = pd.DataFrame([{
        "Scenario": f"Run {run_id}",
        "Total Population": total_pop, 
        "Total Households": total_hh.round(), 
        "Total Employment": total_emp,
        "Growth Weight": growth_weight,
        "Distribution Weight": distribution_weight,
        "Sum of Population Categories": pop_sum.round(),
        "Sum of Household Income Categories": hhinc_sum.round(),
        "Sum of SF/MF Household Categories": sfmf_sum.round(),
        "Sum of Household Size Categories": hhsize_sum.round(),
        "Sum of Household Work Categories": hhwork_sum.round()
    }])
    new_summary_row.to_csv(summary_file, mode='a', header=False, index=False)

    
    # Select desired columns for MAZ verification
    hh_density_values = (scenario_sed.set_index("MAZ").loc[selected_mazs, "RESHH"] / scenario_sed.set_index("MAZ").loc[selected_mazs, "TOAREA"]).round()
    hh_values = (scenario_sed.set_index("MAZ").loc[selected_mazs, "RESHH"]).round()
    hhinc1_values = (scenario_sed.set_index("MAZ").loc[selected_mazs, "HHINC1"]).round()
    hhinc2_values = (scenario_sed.set_index("MAZ").loc[selected_mazs, "HHINC2"]).round()
    hhinc3_values = (scenario_sed.set_index("MAZ").loc[selected_mazs, "HHINC3"]).round()
    hhinc4_values = (scenario_sed.set_index("MAZ").loc[selected_mazs, "HHINC4"]).round()
    hhinc5_values = (scenario_sed.set_index("MAZ").loc[selected_mazs, "HHINC5"]).round()
    employment_values = (scenario_naics.set_index("MAZ").loc[selected_mazs, "Total"]).round()

    # Append Run ID and Density Values to QAQC Results
    qaqc_results = pd.concat([qaqc_results, pd.DataFrame([[run_id] + list(hh_density_values.values) + [growth_weight] + list(hh_values.values) + list(hhinc1_values.values) 
                                                          + list(hhinc2_values.values) + list(hhinc3_values.values)+ list(hhinc4_values.values)+
                                                          list(hhinc5_values.values)+ list(employment_values.values)], columns=qaqc_results.columns)], ignore_index=True)

    print(f"Run {run_id} completed & logged.")
print("All scenarios completed & logged.")

# **Save QAQC Data**
qaqc_results.to_csv(qaqc_file, index=False)
